In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Libraries loaded successfully")

Libraries loaded successfully


In [22]:
df = pd.read_csv("phishing.csv")
print(df.shape)
df.head()

(11054, 32)


,Index,UsingIP,LongURL,ShortURL,Symbol@,Redirecting//,PrefixSuffix-,SubDomains,HTTPS,DomainRegLen,...,UsingPopupWindow,IframeRedirection,AgeofDomain,DNSRecording,WebsiteTraffic,PageRank,GoogleIndex,LinksPointingToPage,StatsReport,class
0,0,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
1,1,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
2,2,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
3,3,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1
4,4,-1,0,-1,1,-1,-1,1,1,-1,...,1,1,1,1,1,-1,1,-1,-1,1


In [23]:
print(df.columns.tolist())

['Index', 'UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//', 'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'Favicon', 'NonStdPort', 'HTTPSDomainURL', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler', 'InfoEmail', 'AbnormalURL', 'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick', 'UsingPopupWindow', 'IframeRedirection', 'AgeofDomain', 'DNSRecording', 'WebsiteTraffic', 'PageRank', 'GoogleIndex', 'LinksPointingToPage', 'StatsReport', 'class']


In [24]:
print(df['class'].value_counts())

class
 1    6157
-1    4897
Name: count, dtype: int64


In [25]:
X = df.drop(['Index', 'class'], axis=1)
y = df['class']

print("Features shape:", X.shape)
print("Label shape:", y.shape)

Features shape: (11054, 30)
Label shape: (11054,)


In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 8843
Testing samples: 2211


In [27]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [28]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.9692446856625961
Precision: 0.9686746987951808
Recall: 0.9765182186234818
F1 Score: 0.9725806451612903


In [29]:
import joblib

joblib.dump(model, 'phishing_model.pkl')
print("Model saved successfully")

Model saved successfully


In [30]:
df2 = pd.read_csv("malicious_phish.csv")
print(df2.shape)
df2.head()

(651191, 2)


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [31]:
print(df2['type'].value_counts())

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


In [32]:
# Keep only benign and phishing
df2 = df2[df2['type'].isin(['benign', 'phishing'])]

# Balance the dataset - sample benign down to match phishing count
phishing_count = df2[df2['type'] == 'phishing'].shape[0]
benign_sample = df2[df2['type'] == 'benign'].sample(n=phishing_count, random_state=42)
phishing_sample = df2[df2['type'] == 'phishing']

df2_balanced = pd.concat([benign_sample, phishing_sample]).reset_index(drop=True)

print(df2_balanced['type'].value_counts())
print(df2_balanced.shape)

type
benign      94111
phishing    94111
Name: count, dtype: int64
(188222, 2)


In [33]:
import re

def extract_features(url):
    features = {}
    
    features['url_length'] = len(url)
    features['has_https'] = 1 if url.startswith('https') else 0
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_at'] = url.count('@')
    features['num_question_marks'] = url.count('?')
    features['num_equal'] = url.count('=')
    features['num_underscore'] = url.count('_')
    features['num_slash'] = url.count('/')
    features['has_ip'] = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_subdomains'] = url.count('.') - 1 if url.count('.') > 1 else 0
    
    return features

# Test it on one URL first
sample_url = df2_balanced['url'].iloc[0]
print("Sample URL:", sample_url)
print(extract_features(sample_url))

Sample URL: montreal.louer.com/ahuntsic-rentals/houses-rent/
{'url_length': 48, 'has_https': 0, 'num_dots': 2, 'num_hyphens': 2, 'num_at': 0, 'num_question_marks': 0, 'num_equal': 0, 'num_underscore': 0, 'num_slash': 3, 'has_ip': 0, 'num_digits': 0, 'num_subdomains': 1}


In [34]:
feature_list = df2_balanced['url'].apply(extract_features)
features_df = pd.DataFrame(feature_list.tolist())

features_df['label'] = df2_balanced['type'].apply(lambda x: 1 if x == 'benign' else -1)

print(features_df.shape)
features_df.head()

(188222, 13)


,url_length,has_https,num_dots,num_hyphens,num_at,num_question_marks,num_equal,num_underscore,num_slash,has_ip,num_digits,num_subdomains,label
0,48,0,2,2,0,0,0,0,3,0,0,1,1
1,44,0,3,0,0,0,0,0,3,0,0,2,1
2,31,0,1,0,0,1,1,0,1,0,0,0,1
3,30,0,1,0,0,0,0,0,3,0,0,0,1
4,32,0,1,1,0,0,0,0,2,0,0,0,1


In [35]:
X2 = features_df.drop('label', axis=1)
y2 = features_df['label']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X2_train, y2_train)

print("Model 2 trained successfully")

Model 2 trained successfully


In [36]:
y2_pred = model2.predict(X2_test)

accuracy2 = accuracy_score(y2_test, y2_pred)
precision2 = precision_score(y2_test, y2_pred)
recall2 = recall_score(y2_test, y2_pred)
f1_2 = f1_score(y2_test, y2_pred)

print("Accuracy:", accuracy2)
print("Precision:", precision2)
print("Recall:", recall2)
print("F1 Score:", f1_2)

Accuracy: 0.8815778987913402
Precision: 0.8872233345960279
Recall: 0.8735148382971922
F1 Score: 0.8803157216494846


In [37]:
sample_size = 1500  # 1500 phishing + 1500 benign = 3000 total

phishing_sample2 = df2_balanced[df2_balanced['type'] == 'phishing'].sample(n=sample_size, random_state=42)
benign_sample2 = df2_balanced[df2_balanced['type'] == 'benign'].sample(n=sample_size, random_state=42)

enrichment_df = pd.concat([phishing_sample2, benign_sample2]).reset_index(drop=True)
print(enrichment_df.shape)
enrichment_df.head()

(3000, 2)


,url,type
0,solucaoassessoriaservicos.com,phishing
1,www.bom.gov.au/marine/,phishing
2,https://meganmacylesolutions.com/secure/login....,phishing
3,https://docs.google.com/spreadsheet/viewform?f...,phishing
4,jakarta.apache.org/bsf/,phishing


In [38]:
import whois
import ssl
import socket
from urllib.parse import urlparse
from datetime import datetime, timezone
from dateutil import parser as date_parser
from rapidfuzz import fuzz
import time

KNOWN_BRANDS = ["paytm", "phonepe", "googlepay", "amazon", "flipkart", "paypal", "fedex", "bookmyshow"]

def get_root_domain(domain):
    parts = domain.split(".")
    if len(parts) > 2:
        return ".".join(parts[-2:])
    return domain

def get_domain_age_days(domain):
    try:
        info = whois.whois(domain)
        creation_date = info.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        if creation_date is None:
            return -1
        if isinstance(creation_date, str):
            creation_date = date_parser.parse(creation_date)
        if creation_date.tzinfo is not None:
            now = datetime.now(timezone.utc)
        else:
            now = datetime.now()
        age = (now - creation_date).days
        return age
    except Exception:
        return -1

def check_https_valid(domain):
    try:
        ctx = ssl.create_default_context()
        with ctx.wrap_socket(socket.socket(), server_hostname=domain) as s:
            s.settimeout(3)
            s.connect((domain, 443))
        return 1
    except Exception:
        return 0

def check_lookalike(domain):
    best_score = 0
    for brand in KNOWN_BRANDS:
        score = fuzz.partial_ratio(brand, domain.lower())
        if score > best_score:
            best_score = score
    return best_score

print("Enrichment functions ready")

Enrichment functions ready


In [39]:
enriched_features = []

for i, row in enrichment_df.iterrows():
    url = row['url']
    label = 1 if row['type'] == 'benign' else -1
    
    # Add protocol if missing (dataset URLs often don't have http/https prefix)
    full_url = url if url.startswith('http') else 'http://' + url
    domain = urlparse(full_url).netloc
    root_domain = get_root_domain(domain)
    
    basic_feats = extract_features(url)
    domain_age = get_domain_age_days(root_domain)
    https_valid = check_https_valid(domain)
    lookalike_score = check_lookalike(domain)
    
    combined = {**basic_feats,
                'domain_age_days': domain_age,
                'https_valid': https_valid,
                'lookalike_score': lookalike_score,
                'label': label}
    
    enriched_features.append(combined)
    
    if i % 200 == 0:
        print(f"Processed {i}/{len(enrichment_df)}")

enriched_df = pd.DataFrame(enriched_features)
print("Enrichment complete!")
print(enriched_df.shape)

Processed 0/3000


Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out


KeyboardInterrupt: 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def get_domain_age_days_safe(domain):
    try:
        info = whois.whois(domain)
        creation_date = info.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        if creation_date is None:
            return -1
        if isinstance(creation_date, str):
            creation_date = date_parser.parse(creation_date)
        if creation_date.tzinfo is not None:
            now = datetime.now(timezone.utc)
        else:
            now = datetime.now()
        return (now - creation_date).days
    except Exception:
        return -1

def check_https_valid_safe(domain):
    try:
        ctx = ssl.create_default_context()
        with ctx.wrap_socket(socket.socket(), server_hostname=domain) as s:
            s.settimeout(2)
            s.connect((domain, 443))
        return 1
    except Exception:
        return 0

enriched_features = []

for i, row in enrichment_df.iterrows():
    try:
        url = row['url']
        label = 1 if row['type'] == 'benign' else -1
        
        full_url = url if url.startswith('http') else 'http://' + url
        domain = urlparse(full_url).netloc
        root_domain = get_root_domain(domain)
        
        basic_feats = extract_features(url)
        domain_age = get_domain_age_days_safe(root_domain)
        https_valid = check_https_valid_safe(domain)
        lookalike_score = check_lookalike(domain)
        
        combined = {**basic_feats,
                    'domain_age_days': domain_age,
                    'https_valid': https_valid,
                    'lookalike_score': lookalike_score,
                    'label': label}
        
        enriched_features.append(combined)
        
    except Exception:
        continue
    
    if i % 200 == 0:
        print(f"Processed {i}/{len(enrichment_df)}")

enriched_df = pd.DataFrame(enriched_features)
print("Enrichment complete!")
print(enriched_df.shape)

In [ ]:
sample_size = 300
phishing_sample2 = df2_balanced[df2_balanced['type'] == 'phishing'].sample(n=sample_size, random_state=42)
benign_sample2 = df2_balanced[df2_balanced['type'] == 'benign'].sample(n=sample_size, random_state=42)
enrichment_df = pd.concat([phishing_sample2, benign_sample2]).reset_index(drop=True)

enriched_features = []
for i, row in enrichment_df.iterrows():
    try:
        url = row['url']
        label = 1 if row['type'] == 'benign' else -1
        full_url = url if url.startswith('http') else 'http://' + url
        domain = urlparse(full_url).netloc

        basic_feats = extract_features(url)
        https_valid = check_https_valid_safe(domain)
        lookalike_score = check_lookalike(domain)

        combined = {**basic_feats, 'https_valid': https_valid,
                    'lookalike_score': lookalike_score, 'label': label}
        enriched_features.append(combined)
    except Exception:
        continue
    if i % 50 == 0:
        print(f"Processed {i}/{len(enrichment_df)}")

enriched_df = pd.DataFrame(enriched_features)
print("Enrichment complete!")
print(enriched_df.shape)

In [40]:
sample_size = 300
phishing_sample2 = df2_balanced[df2_balanced['type'] == 'phishing'].sample(n=sample_size, random_state=42)
benign_sample2 = df2_balanced[df2_balanced['type'] == 'benign'].sample(n=sample_size, random_state=42)
enrichment_df = pd.concat([phishing_sample2, benign_sample2]).reset_index(drop=True)

enriched_features = []
for i, row in enrichment_df.iterrows():
    try:
        url = row['url']
        label = 1 if row['type'] == 'benign' else -1
        full_url = url if url.startswith('http') else 'http://' + url
        domain = urlparse(full_url).netloc

        basic_feats = extract_features(url)
        https_valid = check_https_valid_safe(domain)
        lookalike_score = check_lookalike(domain)

        combined = {**basic_feats, 'https_valid': https_valid,
                    'lookalike_score': lookalike_score, 'label': label}
        enriched_features.append(combined)
    except Exception:
        continue
    if i % 50 == 0:
        print(f"Processed {i}/{len(enrichment_df)}")

enriched_df = pd.DataFrame(enriched_features)
print("Enrichment complete!")
print(enriched_df.shape)

Processed 0/600
Processed 50/600
Processed 100/600
Processed 150/600
Processed 200/600
Processed 250/600
Processed 300/600
Processed 350/600
Processed 400/600
Processed 450/600
Processed 500/600
Processed 550/600
Enrichment complete!
(600, 15)


In [41]:
X3 = enriched_df.drop('label', axis=1)
y3 = enriched_df['label']

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)

model3 = RandomForestClassifier(n_estimators=100, random_state=42)
model3.fit(X3_train, y3_train)

y3_pred = model3.predict(X3_test)

accuracy3 = accuracy_score(y3_test, y3_pred)
precision3 = precision_score(y3_test, y3_pred)
recall3 = recall_score(y3_test, y3_pred)
f1_3 = f1_score(y3_test, y3_pred)

print("Accuracy:", accuracy3)
print("Precision:", precision3)
print("Recall:", recall3)
print("F1 Score:", f1_3)

Accuracy: 0.8083333333333333
Precision: 0.8571428571428571
Recall: 0.7241379310344828
F1 Score: 0.7850467289719626


In [42]:
import joblib

joblib.dump(model2, 'phishing_model_v2.pkl')
print("Model 2 saved successfully")

Model 2 saved successfully
